### Draw main graph

In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import *

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])


logits = model(hparams, X)['logits']
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.Variable(1e-4, name='learning_rate')

if mode == "pretrain":
    train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)
elif mode == "finetune":
    optimizer = tf.train.AdamOptimizer(learning_rate)
    output_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='linear1|linear2')
    train_step = optimizer.minimize(loss, var_list=output_vars, global_step=global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
graph create


### Load model if exist && TensorboardX Logger

In [2]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = save_dir ='../save_model'

#只恢复transformer部分的参数
sess.run(tf.global_variables_initializer())
ref_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='transformer')
saver = tf.train.Saver(ref_vars)

restore_file = tf.train.latest_checkpoint(load_dir)
print(restore_file)
if restore_file is not None:
    saver.restore(sess, restore_file)
    print("Model restored.", restore_file)
else:
    print('model not exist.')

#Logger
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)  
        

../save_model/checkpoint-3500
INFO:tensorflow:Restoring parameters from ../save_model/checkpoint-3500
Model restored. ../save_model/checkpoint-3500


### Train

In [3]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from time import sleep
import time
import math

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train', 0, type, EventDim)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets,
                                                     learning_rate: 1e-4})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 100 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
0	 9.2331085	 10230.293211982633	
Model saved in path: ../save_model/checkpoint-0
10	 6.7759414	 876.5040911227634	
20	 6.0263586	 414.20399931484434	
30	 5.694366	 297.18830998600873	
40	 5.41098	 223.85090433845608	
50	 5.1465287	 171.83397016032657	
60	 5.284117	 197.18004035986738	
70	 5.14912	 172.27979219383218	
80	 5.253228	 191.18244614949066	
90	 5.202319	 181.69312638276887	
100	 5.278625	 196.10005456322284	
Model saved in path: ../save_model/checkpoint-100
110	 5.3166065	 203.69148518668743	
120	 5.311741	 202.70280169542212	
130	 5.0329523	 153.38518530071335	
140	 5.253318	 191.1995855418451	
150	 4.95556	 141.96211232211962	
160	 5.092011	 162.71675256163857	
170	 4.875761	 131.07386668403532	
180	 5.01419	 150.53418435841226	
190	 5.02695	 152.46725970014697	
200	 4.8863525	 132.4695144143977	
Model saved in path: ../save_model/checkpoint-200
210	 4.953034	 141.6039284410848	
220	 5.0027356	 148.81971617188225	
230	 4.969303	 143.92655

KeyboardInterrupt: 

### Compute perplexity on test set

In [5]:
import math

inputs = []
targets = []

l = len(data_test_files)

for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files, 'test', i, 'music', EventDim)
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
    
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

27.86629378729664